# 02 · E1: línea base con un encoder clínico ajustado

### Qué es este experimento

La literatura internacional coincide en que, para **reconocer entidades clínicas (NER)**, la estrategia que mejor funciona es tomar un modelo tipo BERT ya preentrenado con texto clínico y **ajustarlo** (*fine-tuning*) con ejemplos anotados de la tarea. Esto es E1. Es la vara con la que mediremos todo lo demás (los LLM en E2, la destilación en E4).

**Analogía:** un médico recién egresado (el modelo preentrenado) ya "sabe medicina". El ajuste fino es su beca de especialidad: con casos reales anotados por expertos, aprende a hacer muy bien una tarea concreta, en este caso subrayar enfermedades, fármacos y abreviaturas en derivaciones chilenas.

### Qué comparamos

| Modelo | Preentrenado con | Representa |
|---|---|---|
| `PlanTL-GOB-ES/bsc-bio-ehr-es` | Textos biomédicos y clínicos de España (BSC) | El estado del arte en español "de manual" |
| `plncmm/roberta-clinical-wl-es` | Lo anterior + derivaciones chilenas de listas de espera (PLN@CMM, U. de Chile) | El mismo modelo con "pasantía" en texto chileno |

Si el modelo chileno gana, sobre todo en las entidades abreviadas, es evidencia de que **adaptar el modelo al registro local importa**.

### Conexión con el curso

- **Unidad 2:** el ciclo de entrenamiento está escrito a mano (lotes, pérdida, retropropagación, optimizador AdamW, *early stopping*).
- **Unidad 3:** *transfer learning*, porque reutilizamos un modelo preentrenado.
- **Unidad 4:** el modelo es un Transformer de tipo *encoder*, y lo evaluamos con F1 estricto y laxo e intervalos por *bootstrap*.
- **Unidad 7:** los encoders se preentrenaron de forma autosupervisada (lenguaje enmascarado), y comparamos el efecto de la adaptación de dominio.

In [ ]:
from pathlib import Path
import sys, json, time
RAIZ = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(RAIZ / "src"))

import yaml
import pandas as pd
import matplotlib.pyplot as plt
from datos.cwlc import cargar_split
from dataclasses import asdict
from modelos.e1_encoder import Config, Codificador, etiquetas_palabra, entrenar, predecir, evaluar, guardar_resultados, dispositivo
from evaluacion import metricas
from reproducibilidad import huella_completa, verificar_huellas_datos, agregar_semillas

cfg_global = yaml.safe_load(open(RAIZ / "configs" / "experimentos.yaml"))
carpeta = RAIZ / "data" / "raw" / "cwlc"
train, dev, test = (cargar_split(carpeta, s) for s in ["train", "dev", "test"])
print(f"train={len(train)}  dev={len(dev)}  test={len(test)}  | dispositivo: {dispositivo()}")

# Huella digital: los archivos en disco deben ser idénticos a los registrados en configs/huellas_datos.json
ok, difs = verificar_huellas_datos(carpeta, "cwlc")
assert ok, f"¡Los datos cambiaron respecto de los registrados! Archivos distintos: {difs}"
print("Huellas de datos verificadas: el corpus es exactamente el auditado en el notebook 01b.")

## 1. Por qué un modelo por tipo de entidad

En el CWLC las entidades se **anidan**: en "pcte con DM2", la palabra "DM2" es a la vez una *Enfermedad* y una *Abreviatura*. Un etiquetador normal le pone **una** etiqueta a cada palabra, así que no podría marcar ambas.

La solución que usaron los autores del corpus (Báez et al., 2022; Rojas et al., 2022) es entrenar **un etiquetador independiente por tipo**. Cada uno solo aprende a responder "¿esta palabra es parte de una Enfermedad?" (o de un Fármaco, o de una Abreviatura). Hacemos lo mismo para poder comparar.

Las etiquetas usan el esquema **BIO**: `B-` empieza una entidad, `I-` la continúa y `O` es todo lo demás.

In [ ]:
o = next(x for x in train if any(e[3] == "Disease" for e in x.entidades) and any(e[3] == "Abbreviation" for e in x.entidades))
pd.DataFrame({"palabra": o.tokens,
              "Disease": etiquetas_palabra(o, "Disease"),
              "Abbreviation": etiquetas_palabra(o, "Abbreviation")}).T

## 2. Palabras y subpalabras

El modelo no ve palabras: ve **subpalabras** (*tokens*). Las palabras poco frecuentes, como muchas abreviaturas chilenas, se parten en pedazos. Solo la **primera subpalabra** de cada palabra lleva la etiqueta; las demás reciben `-100`, el valor que PyTorch usa para decir "no evalúes esta posición".

Fíjense en cómo cada modelo parte "pcte c/ DM2 desc". Cuantos más pedazos, más le cuesta al modelo reconocer el concepto.

In [ ]:
for nombre in cfg_global["encoders"].values():
    cod = Codificador(nombre, "Disease", 512)
    enc = cod.codificar(["pcte", "c/", "DM2", "desc"], ["O", "O", "B-Disease", "I-Disease"])
    subp = cod.tok.convert_ids_to_tokens(enc["input_ids"])
    print(f"{nombre}\n  subpalabras: {subp}\n  etiquetas:   {enc['labels']}\n")

## 3. Configuración del experimento

- **`MODO = "rapido"`** entrena con 800 oraciones y 2 épocas, para comprobar que todo funciona (unos minutos).
- **`MODO = "completo"`** usa las 8.014 oraciones de entrenamiento y hasta 5 épocas con *early stopping*. Esta es la versión que se reporta.

Hiperparámetros, desde `configs/experimentos.yaml`:

- **Tasa de aprendizaje 3×10⁻⁵ con calentamiento lineal:** los pesos preentrenados se mueven poco, para no "olvidar" lo aprendido.
- **Lote de 16 oraciones.**
- **Hasta 5 épocas:** el entrenamiento se detiene si el F1 en *dev* no mejora en 2 épocas seguidas.

El *test* se usa **una sola vez**, al final.

### Tres semillas

El azar entra en el ajuste fino por dos lados: los pesos iniciales de la capa de clasificación (que se crea nueva) y el orden en que se barajan los lotes. Con datos acotados, solo cambiar la semilla puede mover el F1 varios puntos (Dodge et al., 2020; Mosbach et al., 2021). Por eso **cada combinación modelo × entidad se entrena con 3 semillas (42, 43 y 44)** y se reporta **media ± desviación estándar**. Una diferencia entre modelos menor que esa variación no es una diferencia real.

Además se fija todo lo fijable (`reproducibilidad.fijar_semilla_total`): `random`, NumPy, PyTorch y algoritmos deterministas. En CPU, la misma semilla da exactamente las mismas predicciones (lo comprobamos comparando huellas). En la GPU del Mac (MPS) puede haber diferencias mínimas entre corridas con la misma semilla; las 3 semillas absorben esa variación.

### Huella de cada resultado

Cada JSON de resultados guarda una **huella**: SHA-256 de los datos y de la configuración, el commit de git (y si había cambios sin guardar), la revisión exacta del modelo en Hugging Face, las versiones de los paquetes y el dispositivo. Con eso, cualquiera puede verificar qué produjo cada número.

In [ ]:
MODO = "rapido"          # "rapido" para probar | "completo" para reportar
MODELOS = cfg_global["encoders"]                       # {"espanol_clinico": ..., "chileno_clinico": ..., "general": ...}
MODELOS = {k: v for k, v in MODELOS.items() if k in ["espanol_clinico", "chileno_clinico"]}
TIPOS = cfg_global["entidades"]                        # ["Disease", "Medication", "Abbreviation"]
ent = cfg_global["entrenamiento_encoder"]

SEMILLAS = cfg_global["semillas"]                       # [42, 43, 44]

def config_para(modelo, tipo, semilla):
    if MODO == "rapido":
        return Config(modelo=modelo, tipo=tipo, epocas=2, lr=5e-5, batch=ent["batch"], max_len=ent["max_len"],
                      max_train=800, semilla=semilla)
    return Config(modelo=modelo, tipo=tipo, epocas=ent["epocas"], lr=ent["lr"], batch=ent["batch"],
                  max_len=ent["max_len"], semilla=semilla)

SALIDA = RAIZ / "results" / f"e1_{MODO}"
print(MODO, "|", list(MODELOS), "|", TIPOS, "| semillas", SEMILLAS, "->", SALIDA)
print(f"Entrenamientos a correr: {len(MODELOS) * len(TIPOS) * len(SEMILLAS)}")

## 4. Entrenar y evaluar

Para cada modelo y cada tipo de entidad, la celda:

1. Ajusta el modelo con *train* y elige la mejor época según el F1 en *dev*.
2. Predice sobre *test*.
3. Calcula las métricas: precisión, *recall* y F1 **estricto** (inicio, fin y tipo exactos) y **laxo** (basta con superponerse), con intervalo de confianza del 95% por *bootstrap*. Además, el *recall* en entidades **abreviadas frente a no abreviadas**.
4. Calcula dos **análisis de sensibilidad por fuga** (ver notebook 01b): el F1 en el test **sin las oraciones repetidas en train**, y el *recall* en menciones **ya vistas** en train frente a menciones **nuevas**.
5. Guarda los resultados, con su huella, en `results/` (los JSON de métricas sí se suben a git; las predicciones no).

Si una combinación ya tiene su JSON, se salta: así se puede interrumpir y retomar.

**Entrenamiento largo:** la celda activa `caffeinate` automáticamente para que el Mac no se duerma, y lo apaga al terminar. Hay que dejar el Mac **enchufado, con la tapa abierta y con VS Code abierto**. Si igual se corta, basta volver a correr la celda: solo se repite el entrenamiento que quedó a medias.

En el Mac M5 con MPS, el modo completo toma del orden de minutos por época y por modelo. La primera corrida registra los tiempos reales en el historial.

In [ ]:
import pickle
from sistema import mantener_despierto, dejar_dormir
despierto = mantener_despierto()          # el Mac no se duerme mientras corre esta celda (solo macOS)

CACHE = RAIZ / "results" / "cache" / f"e1_{MODO}"          # predicciones (no van a git)
CACHE.mkdir(parents=True, exist_ok=True)
predicciones = {}
for clave, nombre in MODELOS.items():
    for tipo in TIPOS:
        for semilla in SEMILLAS:
            ruta = SALIDA / f"{clave}_{tipo}_s{semilla}.json"
            ruta_pred = CACHE / f"{clave}_{tipo}_s{semilla}.pkl"
            if ruta.exists() and ruta_pred.exists():
                predicciones[(clave, tipo, semilla)] = pickle.loads(ruta_pred.read_bytes())
                print(f"(ya existe) {ruta.name}"); continue
            print(f"\n=== {clave} · {tipo} · semilla {semilla} ===")
            cfg = config_para(nombre, tipo, semilla)
            t0 = time.time()
            modelo, cod, historial = entrenar(cfg, train, dev)
            pred = predecir(modelo, cod, test, tipo, disp=dispositivo())
            gold = [e for o in test for e in o.entidades if e[3] == tipo]
            res = evaluar(gold, pred, test, tipo, train=train)
            res["clave"], res["minutos_total"] = clave, round((time.time() - t0) / 60, 1)
            huella = huella_completa(asdict(cfg), carpeta, modelo=nombre, splits={"train": train, "dev": dev, "test": test})
            guardar_resultados(ruta, cfg, historial, res, huella)
            ruta_pred.write_bytes(pickle.dumps(pred))
            predicciones[(clave, tipo, semilla)] = pred
            print(f"  F1 estricto test = {res['estricto']['f1']:.3f}  | sin duplicados = {res['test_sin_duplicados']['f1_estricto']:.3f}")
            del modelo
dejar_dormir(despierto)

## 5. Resultados

La tabla junta las métricas de cada combinación de modelo y tipo de entidad. Cómo leerla:

- **F1 estricto:** la métrica principal. Exige marcar exactamente las mismas palabras que el médico.
- **F1 laxo:** perdona errores de límites (por ejemplo, marcar "diabetes mellitus" en vez de "diabetes mellitus tipo 2"). La diferencia entre laxo y estricto mide cuántos errores son solo de borde.
- **Recall abreviadas / no abreviadas:** de las entidades que el médico marcó, qué proporción encontró el modelo en cada grupo (hipótesis H3).
- **F1 sin duplicados:** si cae mucho respecto del F1 completo, parte del rendimiento venía de oraciones repetidas entre train y test.
- **Recall vistas / nuevas:** cuánto del acierto es "memoria" de menciones ya vistas y cuánto es generalización real.

Primero va la tabla por semilla (todas las corridas) y luego la agregada (media ± DE de las 3 semillas), que es la que se reporta.

In [ ]:
filas = []
for f in sorted(SALIDA.glob("*_s*.json")):
    d = json.loads(f.read_text())
    r, c = d["resultados"], d["config"]
    filas.append({"modelo": r["clave"], "tipo": c["tipo"], "semilla": c["semilla"],
                  "P": r["estricto"]["precision"], "R": r["estricto"]["recall"], "F1 estricto": r["estricto"]["f1"],
                  "IC95": f"{r['estricto']['ic95'][0]:.3f}–{r['estricto']['ic95'][1]:.3f}", "F1 laxo": r["laxo"]["f1"],
                  "F1 sin dup.": r["test_sin_duplicados"]["f1_estricto"],
                  "recall abrev.": r["abreviadas"]["recall_estricto"], "recall no abrev.": r["no_abreviadas"]["recall_estricto"],
                  "recall vistas": r["menciones_vistas"]["recall_estricto"], "recall nuevas": r["menciones_nuevas"]["recall_estricto"],
                  "n nuevas": r["menciones_nuevas"]["n_gold"],
                  "épocas": len([h for h in d["historial"] if "epoca" in h]), "min": r.get("minutos_total"),
                  "commit": (d["huella"]["codigo"]["commit"] or "")[:7], "huella pred.": r["huella_predicciones"][:8]})
por_semilla = pd.DataFrame(filas).round(3)
por_semilla

In [ ]:
METRICAS = ["F1 estricto", "F1 laxo", "F1 sin dup.", "recall abrev.", "recall no abrev.", "recall vistas", "recall nuevas"]
def media_de(x):
    a = agregar_semillas(list(x))
    return f"{a['media']:.3f} ± {a['de']:.3f}"
resultados = por_semilla.groupby(["modelo", "tipo"])[METRICAS].agg(media_de)
resultados["n semillas"] = por_semilla.groupby(["modelo", "tipo"]).size()
resultados

In [ ]:
fig, ax = plt.subplots(figsize=(8, 3.5))
g = por_semilla.groupby(["tipo", "modelo"])["F1 estricto"]
g.mean().unstack().plot.bar(ax=ax, rot=0, yerr=g.std().unstack(), capsize=4)   # barras de error = DE entre semillas
ax.set_ylabel("F1 estricto (test)"); ax.set_ylim(0, 1); ax.set_title("E1 · Encoder ajustado por tipo de entidad")
ax.legend(title="", loc="lower right"); plt.tight_layout(); plt.show()

## 6. ¿El modelo chileno es mejor que el español?

Comparamos ambos modelos sobre **las mismas oraciones** con *bootstrap pareado*: se remuestrean las oraciones del test 1.000 veces y se calcula la diferencia de F1 en cada remuestreo. Si el intervalo del 95% de la diferencia **no incluye el 0**, la diferencia es significativa.

Lo hacemos **por semilla**: la conclusión solo es robusta si se repite en las 3. Si con una semilla es significativa y con otra no, la diferencia depende del azar del entrenamiento y no se debe afirmar.

In [ ]:
filas = []
for tipo in TIPOS:
    gold = [e for o in test for e in o.entidades if e[3] == tipo]
    for semilla in SEMILLAS:
        if ("chileno_clinico", tipo, semilla) in predicciones and ("espanol_clinico", tipo, semilla) in predicciones:
            a = metricas.por_documento(gold, predicciones[("chileno_clinico", tipo, semilla)], "estricto")
            b = metricas.por_documento(gold, predicciones[("espanol_clinico", tipo, semilla)], "estricto")
            dif, lo, hi = metricas.bootstrap_pareado(a, b)
            filas.append({"tipo": tipo, "semilla": semilla, "dif F1 (chileno − español)": round(dif, 3),
                          "IC95": f"[{lo:+.3f}, {hi:+.3f}]", "significativa": lo > 0 or hi < 0})
comparacion = pd.DataFrame(filas)
display(comparacion)
print(comparacion.groupby("tipo")["significativa"].agg(["sum", "count"]).rename(columns={"sum": "semillas significativas", "count": "de"}))

## 7. Contexto: ¿cómo se compara con la literatura?

Referencias publicadas con el CWLC (no son directamente comparables, porque usan otra arquitectura o todas las entidades juntas):

| Trabajo | Corpus | Resultado |
|---|---|---|
| Báez et al. 2020 | Piloto, 900 derivaciones | F1 por tipo: Abbreviation 0,92 · Disease 0,77 · Medication 0,75 |
| Báez et al. 2022 | 5.000 derivaciones (estos mismos *splits*) | F1 global ≈ 0,80 (citado por Rojas 2022) |
| Rojas et al. 2022 | Otra partición, 7 tipos | F1 global 0,83 (mejor publicado) |

Si nuestros F1 por tipo quedan en ese rango, la línea base es sólida.

## 8. Conclusiones (completar después de la corrida completa)

- F1 estricto por tipo, y cuál es más difícil:
- ¿El modelo chileno supera al español? ¿En qué tipos?
- Recall en abreviadas frente a no abreviadas (H3):
- Variación entre semillas (DE): ¿es menor que las diferencias entre modelos?
- ¿Cuánto cae el F1 sin duplicados? ¿Cambia el orden de los modelos?
- Recall en menciones nuevas vs vistas: ¿cuánto es generalización?
- Recordar: `roberta-clinical-wl-es` pudo ver texto del test en su preentrenamiento (notebook 01b, sección 6); si gana, hay que declararlo como limitación.
- Tiempo de entrenamiento en el Mac:

**Siguiente notebook:** `03_e2_llm_prompting.ipynb`, donde los LLM locales intentan la misma tarea solo con instrucciones.